In [1]:
import os
import sys
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark import StorageLevel
# 1. CẤU HÌNH MÔI TRƯỜNG
MY_JAVA_HOME = r"D:\java\openjdk-17.0.18b8"
MY_HADOOP_HOME = r"D:\java\hadoop-3.4.3"
os.environ["JAVA_HOME"] = MY_JAVA_HOME
os.environ["HADOOP_HOME"] = MY_HADOOP_HOME
os.environ["SPARK_HOME"] = os.path.dirname(pyspark.__file__)
sys.path.append(os.path.join(MY_HADOOP_HOME, "bin"))
# 2. KHỞI TẠO SPARK SESSION
spark = SparkSession.builder \
    .appName('MetroPT3_SQL_Analysis_Group16') \
    .master("local[*]") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://10.125.222.18:9000") \
    .config('spark.sql.shuffle.partitions', '8') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()
print("-> Trạng thái: Spark Session cho SQL đã sẵn sàng!")
# 3. ĐỌC DỮ LIỆU SẠCH VÀ TẠO ĐẶC TRƯNG THỜI GIAN
HDFS_CLEAN_FOR_SQL = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
print("-> Đang nạp dữ liệu Parquet từ HDFS...")
df = spark.read.parquet(HDFS_CLEAN_FOR_SQL)
# Cột 'timestamp' đã là dạng thời gian chuẩn, chỉ cần trích xuất thẳng ra giờ, tháng, ngày
df = df.withColumn('hour',    F.hour('timestamp')) \
       .withColumn('month',   F.month('timestamp')) \
       .withColumn('weekday', F.dayofweek('timestamp')) \
       .withColumn('date',    F.to_date('timestamp'))
# 4. LƯU TRỮ VÀO BỘ NHỚ ĐỆM (CACHE) & TẠO VIEW TRUY VẤN
print("-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...")
df.persist(StorageLevel.MEMORY_AND_DISK)
# Kích hoạt hành động (Trigger) để ép Spark thực sự nạp dữ liệu vào RAM
total_rows = df.count()
print(f"-> Cached thành công: {total_rows:,} rows")
# Tạo bảng ảo tên là 'sensor' để sử dụng cú pháp spark.sql()
df.createOrReplaceTempView('sensor')
print("=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===")

-> Trạng thái: Spark Session cho SQL đã sẵn sàng!
-> Đang nạp dữ liệu Parquet từ HDFS...
-> Đang đẩy dữ liệu vào RAM (Persist) để tăng tốc 20 câu query...
-> Cached thành công: 1,516,948 rows
=== SETUP HOÀN TẤT. SẴN SÀNG CHẠY CÁC CÂU QUERY ===


In [2]:
# Q3: PHÂN TÍCH CƯỜNG ĐỘ VẬN HÀNH THEO TỪNG NGÀY TRONG TUẦN
spark.sql('''
    CREATE OR REPLACE TEMP VIEW daily_activity AS
    SELECT
        date,
        timestamp,
        COMP,
        Oil_temperature,
        Motor_current,
        CASE DAYOFWEEK(date)
            WHEN 2 THEN '02. Thứ 2'
            WHEN 3 THEN '03. Thứ 3'
            WHEN 4 THEN '04. Thứ 4'
            WHEN 5 THEN '05. Thứ 5'
            WHEN 6 THEN '06. Thứ 6'
            WHEN 7 THEN '07. Thứ 7'
            WHEN 1 THEN '01. Chủ Nhật'
        END AS ngay_trong_tuan,
        -- Khởi động = chuyển từ trạng thái nghỉ sang chạy (1 -> 0)
        CASE
            WHEN COMP = 0
             AND LAG(COMP,1,1)
                 OVER ( PARTITION BY date ORDER BY timestamp) = 1
            THEN 1 ELSE 0
        END AS la_khoi_dong
    FROM sensor''')
q3_weekly_profile = spark.sql('''
    SELECT
        ngay_trong_tuan,
        ROUND(
            SUM(la_khoi_dong) /
            COUNT(DISTINCT date)
        ,2) AS trung_binh_lan_khoi_dong,
        ROUND(
            AVG(CASE WHEN COMP = 0 THEN 1 ELSE 0 END) * 100
        ,1) AS ty_le_thoi_gian_chay_pct,
        ROUND(
            AVG(CASE WHEN COMP = 0 THEN Oil_temperature END)
        ,2) AS nhiet_do_dau_tb,
        ROUND(
            AVG(CASE WHEN COMP = 0 THEN Motor_current END)
        ,2) AS dong_dien_tb
    FROM daily_activity
    GROUP BY ngay_trong_tuan
    ORDER BY ngay_trong_tuan
''')
print("\n========== EXECUTION PLAN ==========\n")
q3_weekly_profile.explain(True)
print("\n--- CƯỜNG ĐỘ VẬN HÀNH THEO TỪNG NGÀY TRONG TUẦN ---")
df_q3_weekly = q3_weekly_profile.toPandas()
try:
    display(df_q3_weekly)
except NameError:
    print(df_q3_weekly.to_string(index=False))


========== EXECUTION PLAN ==========

== Parsed Logical Plan ==
'Sort ['ngay_trong_tuan ASC NULLS FIRST], true
+- 'Aggregate ['ngay_trong_tuan], ['ngay_trong_tuan, 'ROUND(('SUM('la_khoi_dong) / 'COUNT(distinct 'date)), 2) AS trung_binh_lan_khoi_dong#714, 'ROUND(('AVG(CASE WHEN ('COMP = 0) THEN 1 ELSE 0 END) * 100), 1) AS ty_le_thoi_gian_chay_pct#715, 'ROUND('AVG(CASE WHEN ('COMP = 0) THEN 'Oil_temperature END), 2) AS nhiet_do_dau_tb#716, 'ROUND('AVG(CASE WHEN ('COMP = 0) THEN 'Motor_current END), 2) AS dong_dien_tb#717]
   +- 'UnresolvedRelation [daily_activity], [], false

== Analyzed Logical Plan ==
ngay_trong_tuan: string, trung_binh_lan_khoi_dong: double, ty_le_thoi_gian_chay_pct: double, nhiet_do_dau_tb: double, dong_dien_tb: double
Sort [ngay_trong_tuan#725 ASC NULLS FIRST], true
+- Aggregate [ngay_trong_tuan#725], [ngay_trong_tuan#725, round((cast(sum(la_khoi_dong#726) as double) / cast(count(distinct date#720) as double)), 2) AS trung_binh_lan_khoi_dong#714, round((avg(CASE WH

,ngay_trong_tuan,trung_binh_lan_khoi_dong,ty_le_thoi_gian_chay_pct,nhiet_do_dau_tb,dong_dien_tb
0,01. Chủ Nhật,45.67,19.0,68.81,5.76
1,02. Thứ 2,89.39,13.3,63.55,5.69
2,03. Thứ 3,90.48,14.5,62.82,5.36
3,04. Thứ 4,108.87,20.0,66.95,5.49
4,05. Thứ 5,64.60,14.0,65.29,5.80
5,06. Thứ 6,107.13,14.7,65.34,5.43
6,07. Thứ 7,53.00,19.1,68.32,5.70
